<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/llamaindex/01_hybrid_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LlamaIndex でハイブリッド検索 (TiDB + EC 商品カタログ)

このノートブックは **LlamaIndex 版チュートリアルの第 1 回** です。
pytidb 版 `tutorials/pytidb/05_hybrid_search.ipynb` と同じ EC 商品カタログを題材に、**ベクトル検索 + 全文検索 (FTS) のハイブリッド**を LlamaIndex で組み立てます。

## 学習目標

- LlamaIndex の `TiDBVectorStore` で作ったテーブルに **`ALTER TABLE ... ADD FULLTEXT INDEX` で FTS インデックスを追加**する
- `BaseRetriever` を継承して TiDB の `fts_match_word()` を呼ぶ **自作 `TiDBFullTextRetriever`** を作る
- LlamaIndex 標準の `QueryFusionRetriever` で 2 つのリトリーバを **RRF / Relative Score** 融合する
- メタデータフィルタ (`category`) を vector retriever 側にだけ掛ける

## 注意

- LlamaIndex 公式の `TiDBVectorStore` 統合自体は **vector-only**。ハイブリッドは自作 retriever との組み合わせで実現します。
- TiDB の FULLTEXT INDEX は **columnar replica が必要**なので、`ADD_COLUMNAR_REPLICA_ON_DEMAND` 句を付けて作成 → 1 分前後待ってから検索クエリが通るようになります。


## 1. 依存パッケージのインストール


In [ ]:
!pip install -q \
    'llama-index>=0.11' \
    llama-index-vector-stores-tidbvector \
    llama-index-embeddings-huggingface \
    llama-index-llms-openai-like \
    sentence-transformers \
    pymysql \
    requests


## 2. TiDB Cloud Zero でクラスタを作成する


In [ ]:
import requests

ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"

def request_zero_instance(tag: str = "llamaindex-hybrid") -> dict:
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]

instance = request_zero_instance(tag="llamaindex-hybrid")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host    : {conn['host']}")
print(f"User    : {conn['username']}")
print(f"Expires : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 3. 接続文字列ヘルパ + データベース作成


In [ ]:
import sqlalchemy

SSL = "ssl_verify_cert=true&ssl_verify_identity=true"

def make_conn_str(database: str = "") -> str:
    u, p, h = conn["username"], conn["password"], conn["host"]
    return f"mysql+pymysql://{u}:{p}@{h}:4000/{database}?{SSL}"

DATABASE_NAME = "shop_demo_li"
with sqlalchemy.create_engine(make_conn_str()).begin() as cx:
    cx.execute(sqlalchemy.text(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}"))
print(f"DB '{DATABASE_NAME}' 準備OK")


## 4. Embedding と LLM の設定


In [ ]:
import os
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

EMBED_DIM = 384
Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-small")

try:
    from google.colab import ai as _colab_ai  # noqa: F401
    from llama_index.core.llms import CustomLLM, CompletionResponse, LLMMetadata
    class ColabAILLM(CustomLLM):
        @property
        def metadata(self) -> LLMMetadata:
            return LLMMetadata(model_name="colab-ai", context_window=8192, num_output=2048)
        def complete(self, prompt: str, **kw): return CompletionResponse(text=_colab_ai.generate_text(prompt))
        def stream_complete(self, prompt: str, **kw):
            yield CompletionResponse(text=_colab_ai.generate_text(prompt))
    Settings.llm = ColabAILLM()
    print("LLM: google.colab.ai")
except ImportError:
    from llama_index.llms.openai_like import OpenAILike
    Settings.llm = OpenAILike(
        model=os.environ.get("LMSTUDIO_MODEL", "google/gemma-4-e2b"),
        api_base=os.environ.get("LMSTUDIO_BASE", "http://localhost:1234/v1"),
        api_key="lm-studio", is_chat_model=True, timeout=120,
    )
    print(f"LLM: LM Studio ({Settings.llm.model})")
print("embed_model:", Settings.embed_model.model_name)


## 5. 商品データの定義 (12 件)

ベクトル検索のための説明文と、メタデータ (カテゴリ・価格・名前) を持たせます。


In [ ]:
PRODUCTS = [
    ("electronics", 12800, "ワイヤレスノイズキャンセリングイヤホン",
     "周囲の騒音を打ち消しながら音楽や通話を楽しめる完全ワイヤレス型のイヤホン。通勤や旅行で集中したい人向け。"),
    ("electronics", 28900, "コンパクトドローン 4K カメラ付き",
     "折りたたんでカバンに入る小型ドローン。屋外で空撮を楽しみたい初心者から中級者に。"),
    ("electronics", 9800,  "スマートポータブル加湿器",
     "USB 給電の小型加湿器。オフィスや枕元で乾燥を防ぎ、静音設計で在宅勤務に最適。"),
    ("kitchen",     4980,  "圧力調理マルチポット",
     "圧力・蒸し・炒めができる一台多役の調理器。忙しい平日の夕食づくりを 30 分で終わらせたい人向け。"),
    ("kitchen",     1980,  "ステンレスコーヒードリッパー",
     "紙フィルター不要の金属メッシュドリッパー。豆の油分まで抽出できて、コーヒーを毎日淹れる人におすすめ。"),
    ("kitchen",     3480,  "電動ミル付きペッパーグラインダー",
     "ボタン一つでスパイスを挽ける電動式グラインダー。料理の仕上げに挽きたての香りを楽しめる。"),
    ("outdoor",     15800, "軽量登山テント 2 人用",
     "総重量 1.9kg の軽量テント。2 人が快適に泊まれるサイズで、縦走やキャンプツーリングに最適。"),
    ("outdoor",     6980,  "防水リュック 30L",
     "完全防水素材を使った 30L のリュック。雨天の通勤や川沿いのアウトドアで中身を守る。"),
    ("outdoor",     2980,  "折りたたみチタンマグカップ",
     "薄くて軽いチタン製のマグカップ。キャンプや登山で温かい飲み物をすぐに楽しめる。"),
    ("fashion",     4280,  "メリノウール ベースレイヤー",
     "冬のアウトドアにも普段着にも使えるメリノウール製の長袖インナー。臭いを抑える天然素材。"),
    ("fashion",     8900,  "撥水ライトシェルジャケット",
     "急な雨や風を防ぐ軽量シェル。自転車通勤やハイキングで羽織るのに便利なアウター。"),
    ("fashion",     1280,  "シンプルコットンTシャツ",
     "肉厚のコットンを使ったユニセックスのTシャツ。普段使いのローテに入れやすいベーシック。"),
]

print(f"商品件数: {len(PRODUCTS)}")


## 6. `Document` に変換して TiDB に投入

`text=description`、`metadata={"category", "price", "name"}` の構成です。
`VectorStoreIndex.from_documents` が embedding を計算 → TiDB に書き込みます。


In [ ]:
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.vector_stores.tidbvector import TiDBVectorStore

documents = [
    Document(
        text=desc,
        metadata={"name": name, "category": cat, "price": price},
    )
    for (cat, price, name, desc) in PRODUCTS
]

vector_store = TiDBVectorStore(
    connection_string=make_conn_str(DATABASE_NAME),
    table_name="products",
    distance_strategy="cosine",
    vector_dimension=EMBED_DIM,
    drop_existing_table=True,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(f"投入完了: {len(documents)} 件")


## 7. 全文検索 (FTS) インデックスの追加

LlamaIndex の `TiDBVectorStore` が作るテーブル列名は **`document`** (TEXT) です。これに `MULTILINGUAL` パーサで FULLTEXT INDEX を追加します。

TiDB の全文索引は **columnar replica** が必要なので `ADD_COLUMNAR_REPLICA_ON_DEMAND` 句を付けます。replica の同期が完了するまで `fts_match_word` がエラーになるため、**インデックス利用可能になるまで数十秒〜1 分待つポーリング**を入れます。


In [ ]:
import time

engine = sqlalchemy.create_engine(make_conn_str(DATABASE_NAME))

with engine.begin() as cx:
    cx.execute(sqlalchemy.text(
        "ALTER TABLE products ADD FULLTEXT INDEX ft_doc (document) "
        "WITH PARSER MULTILINGUAL ADD_COLUMNAR_REPLICA_ON_DEMAND"
    ))
print("FTS index DDL 実行")

# columnar replica の同期完了を TIFLASH_REPLICA 経由で待つ。
# replica 同期が完了 (PROGRESS=1.0) すれば fts_match_word も呼べるようになる。
print("columnar replica 同期待ち", end="", flush=True)
for i in range(180):
    with engine.connect() as cx:
        progress = cx.execute(sqlalchemy.text(
            "SELECT PROGRESS FROM INFORMATION_SCHEMA.TIFLASH_REPLICA "
            "WHERE TABLE_SCHEMA = :db AND TABLE_NAME = \'products\'"
        ), {"db": DATABASE_NAME}).scalar()
    if progress is not None and float(progress) >= 1.0:
        print(f" 完了 (PROGRESS={progress})"); break
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print(" timeout (続行するが空ヒットの可能性あり)")


## 8. `TiDBFullTextRetriever` を作る

`BaseRetriever` を継承して、TiDB の `fts_match_word` 関数で全文検索を行うリトリーバを作ります。
結果を `NodeWithScore(node=TextNode(...), score=...)` で返せば LlamaIndex のリトリーバ群と統一して扱えます。


In [ ]:
import json
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, TextNode


class TiDBFullTextRetriever(BaseRetriever):
    def __init__(self, engine, table: str, top_k: int = 10,
                 text_col: str = "document", meta_col: str = "meta"):
        super().__init__()
        self._engine = engine
        self._table = table
        self._top_k = top_k
        self._text_col = text_col
        self._meta_col = meta_col

    def _retrieve(self, query_bundle):
        sql = sqlalchemy.text(f"""
            SELECT id, {self._text_col} AS body, {self._meta_col} AS meta,
                   fts_match_word(:q, {self._text_col}) AS s
            FROM {self._table}
            WHERE fts_match_word(:q, {self._text_col})
            ORDER BY s DESC
            LIMIT :k
        """)
        with self._engine.connect() as cx:
            rows = cx.execute(sql, {"q": query_bundle.query_str, "k": self._top_k}).all()
        results = []
        for r in rows:
            meta = r.meta if isinstance(r.meta, dict) else (json.loads(r.meta) if r.meta else {})
            # id_=str(r.id) を渡すことで vec_retriever (TiDBVectorStore が割り当てた UUID) と
            # 同じ node_id になり、QueryFusionRetriever が同一ドキュメントとして dedup できる。
            results.append(NodeWithScore(
                node=TextNode(id_=str(r.id), text=r.body, metadata=meta),
                score=float(r.s),
            ))
        return results


fts_retriever = TiDBFullTextRetriever(engine, table="products", top_k=10)
print("TiDBFullTextRetriever 準備OK")


## 9. ベクトル単独 vs 全文単独 を見比べる

同じクエリ「雨の日の通勤が快適になるもの」をそれぞれに投げます。
ベクトル検索は意味の近さ、全文検索は単語一致 (雨・通勤) で順位が決まります。


In [ ]:
QUERY = "雨の日の通勤が快適になるもの"

vec_retriever = index.as_retriever(similarity_top_k=5)

print(f"Query: {QUERY}\n")
print("[ベクトル検索 (LlamaIndex の TiDBVectorStore)]")
for n in vec_retriever.retrieve(QUERY):
    print(f"  sim={n.score:.3f}  [{n.metadata.get('category')}] {n.metadata.get('name')}")

print("\n[全文検索 (TiDBFullTextRetriever, MULTILINGUAL)]")
for n in fts_retriever.retrieve(QUERY):
    print(f"  ms={n.score:.3f}   [{n.metadata.get('category')}] {n.metadata.get('name')}")
    print(f"    {n.node.text}")


## 10. ハイブリッド検索 (RRF: `reciprocal_rerank`)

LlamaIndex の `QueryFusionRetriever` に **2 つのリトリーバ** (ベクトル + FTS) を渡します。

`num_queries=1` を必ず指定してください — デフォルトの `4` だと LLM がクエリ書き換えを行う追加呼び出しが走ります (今回は不要)。


In [ ]:
from llama_index.core.retrievers import QueryFusionRetriever

hybrid_rrf = QueryFusionRetriever(
    [vec_retriever, fts_retriever],
    similarity_top_k=5,
    num_queries=1,                   # クエリ書き換え無効化 (LLM 呼び出し不要)
    mode="reciprocal_rerank",        # RRF
    use_async=False,
)

print(f"[hybrid (RRF)] Query: {QUERY}")
for n in hybrid_rrf.retrieve(QUERY):
    print(f"  score={n.score:.4f}  [{n.metadata.get('category')}] {n.metadata.get('name')}")


## 11. ハイブリッド検索 (重み付き: `relative_score`)

`mode="relative_score"` + `retriever_weights` でベクトル/全文の重みを変えられます。
まずはベクトル寄り (0.7 / 0.3)、その後キーワード寄り (0.3 / 0.7) を試します。


In [ ]:
hybrid_w_vec = QueryFusionRetriever(
    [vec_retriever, fts_retriever],
    retriever_weights=[0.7, 0.3],
    similarity_top_k=5, num_queries=1,
    mode="relative_score", use_async=False,
)
print(f"[hybrid (weighted, vec=0.7/fts=0.3)] Query: {QUERY}")
for n in hybrid_w_vec.retrieve(QUERY):
    print(f"  score={n.score:.4f}  [{n.metadata.get('category')}] {n.metadata.get('name')}")

hybrid_w_fts = QueryFusionRetriever(
    [vec_retriever, fts_retriever],
    retriever_weights=[0.3, 0.7],
    similarity_top_k=5, num_queries=1,
    mode="relative_score", use_async=False,
)
print(f"\n[hybrid (weighted, vec=0.3/fts=0.7)] Query: {QUERY}")
for n in hybrid_w_fts.retrieve(QUERY):
    print(f"  score={n.score:.4f}  [{n.metadata.get('category')}] {n.metadata.get('name')}")


## 12. メタデータフィルタとの併用

LlamaIndex の `MetadataFilters` でカテゴリ絞り込みができます。
TiDBVectorStore は **`==` / `!=` のみ**サポートしているので、`category in ["outdoor", "fashion"]` は **OR 結合**で表現します。

フィルタは vector retriever 側に掛けます (`TiDBFullTextRetriever` は raw SQL 経由なので別途 `WHERE` を付けても良いが、ここでは vector 側だけ絞る運用例)。


In [ ]:
from llama_index.core.vector_stores.types import (
    MetadataFilters, MetadataFilter, FilterCondition,
)

filters = MetadataFilters(
    filters=[
        MetadataFilter(key="category", value="outdoor", operator="=="),
        MetadataFilter(key="category", value="fashion", operator="=="),
    ],
    condition=FilterCondition.OR,
)

vec_retriever_filtered = index.as_retriever(similarity_top_k=10, filters=filters)
hybrid_filtered = QueryFusionRetriever(
    [vec_retriever_filtered, fts_retriever],
    similarity_top_k=5, num_queries=1,
    mode="reciprocal_rerank", use_async=False,
)

Q2 = "山でも普段でも使える"
print(f"[hybrid + filter(category in outdoor/fashion)] Query: {Q2}")
for n in hybrid_filtered.retrieve(Q2):
    print(f"  score={n.score:.4f}  [{n.metadata.get('category')}] {n.metadata.get('name')}")


## 13. チャレンジ

- `vec=0.3 / fts=0.7` の結果と `vec=0.7 / fts=0.3` の結果を見比べて、どちらがどんなクエリに強いか考える
- `mode="relative_score"` を `mode="dist_based_score"` に変えてみる (LlamaIndex の別フュージョン手法)
- クエリを英語にしたとき (`"waterproof commute gear"`) の結果を確認
- `TiDBFullTextRetriever` 側にも `category` フィルタを加える (生 SQL に `WHERE JSON_EXTRACT(meta, '$.category') = 'outdoor'` を追加)

## 補足

- LlamaIndex の `TiDBVectorStore` は今のところ **vector-only** の統合です。今回示した「自作 retriever + QueryFusionRetriever で組み立てる」パターンは、TiDB に限らず他のベクトルストアでも使えます。
- 真の「TiDB 流ハイブリッド検索」は pytidb の `table.search(QUERY, search_type="hybrid")` (`tutorials/pytidb/05_hybrid_search.ipynb`) で 1 行で書けます。SDK 選択の比較材料としてみてください。
